In [ ]:
# Baseline CPU FP32 Inference for PYNQ

import numpy as np
import json
import time

# Load all artifacts
w = {
    "fc1": np.load("artifacts/fc1_weight.npy"), # (128, 784) int8
    "fc2": np.load("artifacts/fc2_weight.npy"), # (64, 128) int8
    "fc3": np.load("artifacts/fc3_weight.npy"), # (10, 64) int8
}
b = {
    "fc1": np.load("artifacts/fc1_bias.npy"),
    "fc2": np.load("artifacts/fc2_bias.npy"),
    "fc3": np.load("artifacts/fc3_bias.npy"),
}
with open("artifacts/weight_scales.json") as f:
    ws = json.load(f)

images = np.load("artifacts/mnist_test_images.npy")
labels = np.load("artifacts/mnist_test_labels.npy")

# Dequantize weights to FP32 once
W = {
    "fc1": w["fc1"].astype(np.float32) * ws["fc1"], # (128, 784)
    "fc2": w["fc2"].astype(np.float32) * ws["fc2"], # (64, 128)
    "fc3": w["fc3"].astype(np.float32) * ws["fc3"], # (10, 64)
}

def run_mlp_one_image(x_fp32):
    x = x_fp32.reshape(1, -1)

    x = np.maximum(x @ W["fc1"].T + b["fc1"], 0.0).astype(np.float32)
    x = np.maximum(x @ W["fc2"].T + b["fc2"], 0.0).astype(np.float32)
    logits = (x @ W["fc3"].T + b["fc3"]).flatten()

    pred = int(np.argmax(logits))
    return pred, logits


print("===== CPU FP32 Predictions =====")
for i in range(5):
    pred, logits = run_mlp_one_image(images[i])
    print(f"Sample {i}: label={labels[i]}, pred={pred}")
    print("  logits =", logits)

correct = 0
for i in range(1000):
    pred, _ = run_mlp_one_image(images[i])
    correct += (pred == labels[i])
print(f"\nAccuracy on first 1000 samples: {correct / 1000:.4f}")

n_samples = 100
start = time.perf_counter()
for i in range(n_samples):
    run_mlp_one_image(images[i])
avg_ms = (time.perf_counter() - start) * 1000.0 / n_samples
print(f"Average CPU FP32 latency: {avg_ms:.4f} ms/sample")


In [ ]:
# Baseline CPU INT8 Inference for PYNQ

import numpy as np
import json
import time

# Load all artifacts
w = {
    "fc1": np.load("artifacts/fc1_weight.npy"), # (128, 784) int8
    "fc2": np.load("artifacts/fc2_weight.npy"), # (64, 128) int8
    "fc3": np.load("artifacts/fc3_weight.npy"), # (10, 64) int8
}
b = {
    "fc1": np.load("artifacts/fc1_bias.npy"),
    "fc2": np.load("artifacts/fc2_bias.npy"),
    "fc3": np.load("artifacts/fc3_bias.npy"),
}
with open("artifacts/weight_scales.json") as f:
    ws = json.load(f)
with open("artifacts/activation_scales.json") as f:
    act_scales = json.load(f)

images = np.load("artifacts/mnist_test_images.npy")
labels = np.load("artifacts/mnist_test_labels.npy")

def quantize_input(x_fp32):
    return np.clip(
        np.round(x_fp32.reshape(-1) / act_scales["input"]),
        -128, 127
    ).astype(np.int8)

def run_mlp_one_image(x_fp32):
    x = quantize_input(x_fp32).reshape(1, -1)

    # FC1
    y = x.astype(np.int32) @ w["fc1"].T.astype(np.int32)
    y = y.astype(np.float32) * (act_scales["input"] * ws["fc1"]) + b["fc1"]
    x = np.clip(np.round(np.maximum(y, 0.0) / act_scales["relu1_out"]), -128, 127).astype(np.int8)

    # FC2
    y = x.astype(np.int32) @ w["fc2"].T.astype(np.int32)
    y = y.astype(np.float32) * (act_scales["relu1_out"] * ws["fc2"]) + b["fc2"]
    x = np.clip(np.round(np.maximum(y, 0.0) / act_scales["relu2_out"]), -128, 127).astype(np.int8)

    # FC3
    y = x.astype(np.int32) @ w["fc3"].T.astype(np.int32)
    logits = (y.astype(np.float32) * (act_scales["relu2_out"] * ws["fc3"]) + b["fc3"]).flatten()

    pred = int(np.argmax(logits))
    return pred, logits


print("===== CPU INT8 Predictions =====")
for i in range(5):
    pred, logits = run_mlp_one_image(images[i])
    print(f"Sample {i}: label={labels[i]}, pred={pred}")
    print("  logits =", logits)

correct = 0
for i in range(1000):
    pred, _ = run_mlp_one_image(images[i])
    correct += (pred == labels[i])
print(f"\nAccuracy on first 1000 samples: {correct / 1000:.4f}")

n_samples = 100
start = time.perf_counter()
for i in range(n_samples):
    run_mlp_one_image(images[i])
avg_ms = (time.perf_counter() - start) * 1000.0 / n_samples
print(f"Average CPU INT8 latency: {avg_ms:.4f} ms/sample")


In [ ]:
# Multi-GEMM Accelerated Inference

import numpy as np
import json
import time
from pynq import Overlay, allocate

# Load all artifacts
w = {
    "fc1": np.load("artifacts/fc1_weight.npy"), # (128, 784) int8
    "fc2": np.load("artifacts/fc2_weight.npy"), # (64, 128) int8
    "fc3": np.load("artifacts/fc3_weight.npy"), # (10, 64) int8
}
b = {
    "fc1": np.load("artifacts/fc1_bias.npy"),
    "fc2": np.load("artifacts/fc2_bias.npy"),
    "fc3": np.load("artifacts/fc3_bias.npy"),
}
with open("artifacts/weight_scales.json") as f:
    ws = json.load(f)
with open("artifacts/activation_scales.json") as f:
    act_scales = json.load(f)

images = np.load("artifacts/mnist_test_images.npy")
labels = np.load("artifacts/mnist_test_labels.npy")

ol = Overlay("block_design.bit")
ip = ol.multiply_0

# Pre-load weight buffers, synced once
layers = [
    {"name": "fc1", "K": 784, "N": 128},
    {"name": "fc2", "K": 128, "N":  64},
    {"name": "fc3", "K":  64, "N":  10},
]

for layer in layers:
    K, N = layer["K"], layer["N"]
    buf = allocate(shape=(K * N,), dtype=np.int8)
    buf[:] = w[layer["name"]].T.flatten()
    buf.sync_to_device()
    layer["B_buf"] = buf

A_buf = allocate(shape=(784,), dtype=np.int8)
C_buf = allocate(shape=(128,), dtype=np.int32)


def quantize_input(x_fp32):
    return np.clip(
        np.round(x_fp32.reshape(-1) / act_scales["input"]),
        -128, 127
    ).astype(np.int8)

def run_fpga_gemm(x_int8, B_buf, K, N):
    A_buf[:K] = x_int8
    A_buf.sync_to_device()
    ip.write(0x10, A_buf.device_address & 0xFFFFFFFF)
    ip.write(0x14, (A_buf.device_address >> 32) & 0xFFFFFFFF)
    ip.write(0x1c, B_buf.device_address & 0xFFFFFFFF)
    ip.write(0x20, (B_buf.device_address >> 32) & 0xFFFFFFFF)
    ip.write(0x28, C_buf.device_address & 0xFFFFFFFF)
    ip.write(0x2c, (C_buf.device_address >> 32) & 0xFFFFFFFF)
    ip.write(0x34, 1)
    ip.write(0x3c, K)
    ip.write(0x44, N)
    ip.write(0x00, 0x01)
    while (ip.read(0x00) & 0x2) == 0:
        pass
    C_buf.sync_from_device()
    return np.array(C_buf[:N], dtype=np.int32)

def run_mlp_one_image(x_fp32):
    x = quantize_input(x_fp32)

    # FC1
    y = run_fpga_gemm(x, layers[0]["B_buf"], K=784, N=128)
    y = y.astype(np.float32) * (act_scales["input"] * ws["fc1"]) + b["fc1"]
    x = np.clip(np.round(np.maximum(y, 0.0) / act_scales["relu1_out"]), -128, 127).astype(np.int8)

    # FC2
    y = run_fpga_gemm(x, layers[1]["B_buf"], K=128, N=64)
    y = y.astype(np.float32) * (act_scales["relu1_out"] * ws["fc2"]) + b["fc2"]
    x = np.clip(np.round(np.maximum(y, 0.0) / act_scales["relu2_out"]), -128, 127).astype(np.int8)

    # FC3
    y = run_fpga_gemm(x, layers[2]["B_buf"], K=64, N=10)
    logits = y.astype(np.float32) * (act_scales["relu2_out"] * ws["fc3"]) + b["fc3"]

    pred = int(np.argmax(logits))
    return pred, logits


print("===== Multi-GEMM FPGA Predictions =====")
for i in range(5):
    pred, logits = run_mlp_one_image(images[i])
    print(f"Sample {i}: label={labels[i]}, pred={pred}")
    print("  logits =", logits)

correct = 0
for i in range(1000):
    pred, _ = run_mlp_one_image(images[i])
    correct += (pred == labels[i])
print(f"\nAccuracy on first 1000 samples: {correct / 1000:.4f}")

n_samples = 100
start = time.perf_counter()
for i in range(n_samples):
    run_mlp_one_image(images[i])
avg_ms = (time.perf_counter() - start) * 1000.0 / n_samples
print(f"Average FPGA latency: {avg_ms:.4f} ms/sample")

A_buf.close()
C_buf.close()
for layer in layers:
    layer["B_buf"].close()


In [ ]:
# Monolithic Only Accelerated Inference

import numpy as np
import json
import time
from pynq import Overlay, allocate

# Load all artifacts
w = {
    "fc1": np.load("artifacts/fc1_weight.npy"), # (128, 784) int8
    "fc2": np.load("artifacts/fc2_weight.npy"), # (64, 128) int8
    "fc3": np.load("artifacts/fc3_weight.npy"), # (10, 64) int8
}
b = {
    "fc1": np.load("artifacts/fc1_bias_int32.npy").astype(np.int32),
    "fc2": np.load("artifacts/fc2_bias_int32.npy").astype(np.int32),
    "fc3": np.load("artifacts/fc3_bias_int32.npy").astype(np.int32),
}

with open("artifacts/activation_scales.json") as f:
    act_scales = json.load(f)

images = np.load("artifacts/mnist_test_images.npy")
labels = np.load("artifacts/mnist_test_labels.npy")

ol = Overlay("monolithic_only.bit")   
ip = ol.mlp_0

A_buf  = allocate(shape=(784,), dtype=np.int8)
W1_buf = allocate(shape=(784 * 128,), dtype=np.int8)
b1_buf = allocate(shape=(128,), dtype=np.int32)
W2_buf = allocate(shape=(128 * 64,), dtype=np.int8)
b2_buf = allocate(shape=(64,), dtype=np.int32)
W3_buf = allocate(shape=(64 * 10,), dtype=np.int8)
b3_buf = allocate(shape=(10,), dtype=np.int32)
C_buf  = allocate(shape=(10,), dtype=np.int32)


W1_buf[:] = w["fc1"].T.reshape(-1)
W2_buf[:] = w["fc2"].T.reshape(-1)
W3_buf[:] = w["fc3"].T.reshape(-1)

b1_buf[:] = b["fc1"]
b2_buf[:] = b["fc2"]
b3_buf[:] = b["fc3"]

W1_buf.sync_to_device()
b1_buf.sync_to_device()
W2_buf.sync_to_device()
b2_buf.sync_to_device()
W3_buf.sync_to_device()
b3_buf.sync_to_device()

def quantize_input(x_fp32):
    x_int8 = np.clip(
        np.round(x_fp32.reshape(-1) / act_scales["input"]),
        -128, 127
    ).astype(np.int8)
    return x_int8

def run_mlp_one_image(x_fp32):
    x_int8 = quantize_input(x_fp32)

    A_buf[:] = x_int8
    A_buf.sync_to_device()

    ip.write(0x10, A_buf.device_address & 0xFFFFFFFF)
    ip.write(0x14, (A_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x1c, W1_buf.device_address & 0xFFFFFFFF)
    ip.write(0x20, (W1_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x28, b1_buf.device_address & 0xFFFFFFFF)
    ip.write(0x2c, (b1_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x34, W2_buf.device_address & 0xFFFFFFFF)
    ip.write(0x38, (W2_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x40, b2_buf.device_address & 0xFFFFFFFF)
    ip.write(0x44, (b2_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x4c, W3_buf.device_address & 0xFFFFFFFF)
    ip.write(0x50, (W3_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x58, b3_buf.device_address & 0xFFFFFFFF)
    ip.write(0x5c, (b3_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x64, C_buf.device_address & 0xFFFFFFFF)
    ip.write(0x68, (C_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x00, 0x01)
    while (ip.read(0x00) & 0x2) == 0:
        pass

    C_buf.sync_from_device()
    logits = np.array(C_buf, dtype=np.int32)
    pred = int(np.argmax(logits))
    return pred, logits


print("===== Monolithic MLP FPGA Predictions =====")
for i in range(5):
    pred, logits = run_mlp_one_image(images[i])
    print(f"Sample {i}: label={labels[i]}, pred={pred}")
    print("  logits =", logits)


correct = 0
for i in range(1000):
    pred, _ = run_mlp_one_image(images[i])
    correct += (pred == labels[i])

print(f"\nAccuracy on first 1000 samples: {correct / 1000:.4f}")


n_samples = 100
start = time.perf_counter()
for i in range(n_samples):
    run_mlp_one_image(images[i])
avg_ms = (time.perf_counter() - start) * 1000.0 / n_samples
print(f"Average FPGA latency: {avg_ms:.4f} ms/sample")


A_buf.close()
W1_buf.close()
b1_buf.close()
W2_buf.close()
b2_buf.close()
W3_buf.close()
b3_buf.close()
C_buf.close()


In [ ]:
# Monolithic Hardcoded Weights Accelerated Inference

import numpy as np
import json
import time
from pynq import Overlay, allocate

# Load all artifacts
with open("artifacts/activation_scales.json") as f:
    act_scales = json.load(f)

images = np.load("artifacts/mnist_test_images.npy")
labels = np.load("artifacts/mnist_test_labels.npy")


ol = Overlay("monolithic_hardcode.bit")   
ip = ol.mlp_0                      


A_buf = allocate(shape=(784,), dtype=np.int8)
C_buf = allocate(shape=(10,), dtype=np.int32)

def quantize_input(x_fp32):
    x_int8 = np.clip(
        np.round(x_fp32.reshape(-1) / act_scales["input"]),
        -128, 127
    ).astype(np.int8)
    return x_int8

def run_mlp_one_image(x_fp32):
    x_int8 = quantize_input(x_fp32)

    A_buf[:] = x_int8
    A_buf.sync_to_device()


    ip.write(0x10, A_buf.device_address & 0xFFFFFFFF)
    ip.write(0x14, (A_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x1c, C_buf.device_address & 0xFFFFFFFF)
    ip.write(0x20, (C_buf.device_address >> 32) & 0xFFFFFFFF)

    ip.write(0x00, 0x01)
    while (ip.read(0x00) & 0x2) == 0:
        pass

    C_buf.sync_from_device()
    logits = np.array(C_buf, dtype=np.int32)
    pred = int(np.argmax(logits))
    return pred, logits


print("===== Hardcoded Monolithic FPGA Predictions =====")
for i in range(5):
    pred, logits = run_mlp_one_image(images[i])
    print(f"Sample {i}: label={labels[i]}, pred={pred}")
    print("  logits =", logits)


correct = 0
for i in range(1000):
    pred, _ = run_mlp_one_image(images[i])
    correct += (pred == labels[i])

print(f"\nAccuracy on first 1000 samples: {correct / 1000:.4f}")


n_samples = 100
start = time.perf_counter()
for i in range(n_samples):
    run_mlp_one_image(images[i])
avg_ms = (time.perf_counter() - start) * 1000.0 / n_samples
print(f"Average FPGA latency: {avg_ms:.4f} ms/sample")


A_buf.close()
C_buf.close()